# Appendix 10.3: Search & Retrieval

**Technique:** Retrieval Augmented Generation (RAG)

LLMs have a fixed context window. When the information a user needs lives in a large corpus (docs, tickets, code, emails), you can't paste all of it into the prompt. **Retrieval augmented generation (RAG)** is the standard solution: fetch only the relevant chunk(s), inject them into the prompt, and tell Claude to answer from that context.

**Contents**
* [Concept: Retrieval Augmented Prompting](#concept)
* [Minimal RAG Example](#example)
* Going Further: Web Search, Fetch, and the Files API
* [Reflect and Congratulations](#reflect)

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Concept: Retrieval Augmented Prompting {#concept}

The pattern has three steps:

```
1. Retrieve: given the user's query, find the most relevant chunk(s) from your corpus
2. Inject:   paste those chunk(s) into the prompt, wrapped in clear XML tags
3. Generate: ask Claude to answer *only* from the supplied context
```

This is a direct application of the **grounding** principle from Chapter 8 (Long Context). Instead of asking Claude to recall facts from training data, you supply a precise excerpt and instruct Claude to reason over it. The model doesn't need to know your internal docs ahead of time. You bring the facts to the prompt at runtime.

### Why XML tags for context?

Claude is trained to treat content inside tags as structured input. A tag like `<context>` makes the boundary between retrieved content and the rest of the prompt unambiguous, so Claude doesn't confuse "what the docs say" with "what the user said."

### Retrieval strategies

* **Keyword / BM25**: matches query terms against document tokens. Best for exact term lookups and is fast.
* **Semantic / embedding**: embeds the query and docs, then runs a cosine similarity search. Best for conceptual queries and synonyms.
* **Hybrid**: keyword reranking on top of semantic search. Best for production RAG pipelines.

For this appendix, a naive keyword filter is enough to illustrate the pattern. Real production systems reach for a vector database (Pinecone, pgvector, Qdrant, …) or a managed search service.

In [ ]:
// Minimal RAG example
// =====================
// 1. A tiny in memory "corpus" of engineering runbook snippets.
// 2. A naive keyword retrieve() that scores each doc by query term overlap.
// 3. A prompt that injects the top result inside <context> tags.

const DOCS = [
  {
    id: "runbook-deploy",
    text: "Deployment runbook: run `npm run build` then `kubectl apply -f k8s/`. " +
          "Verify with `kubectl rollout status deployment/api`. " +
          "Rollback with `kubectl rollout undo deployment/api`.",
  },
  {
    id: "runbook-db",
    text: "Database runbook: take a snapshot with `pg_dump -Fc mydb > backup.dump` before any migration. " +
          "Run migrations with `npx prisma migrate deploy`. " +
          "Restore from backup with `pg_restore -d mydb backup.dump`.",
  },
  {
    id: "runbook-alerts",
    text: "Alerting runbook: PagerDuty escalation policy is 'P1-Engineering'. " +
          "Silence a noisy alert with `amtool silence add`. " +
          "Check alert history in Grafana under Alerting > Alert rules.",
  },
];

/**
 * Naive keyword retrieve: score each doc by how many query words it contains.
 * Returns the single best matching document, or null if nothing matches.
 */
function retrieve(query, docs) {
  const terms = query.toLowerCase().split(/\s+/);
  let best = null;
  let bestScore = 0;
  for (const doc of docs) {
    const lower = doc.text.toLowerCase();
    const score = terms.filter((t) => lower.includes(t)).length;
    if (score > bestScore) {
      bestScore = score;
      best = doc;
    }
  }
  return bestScore > 0 ? best : null;
}

// === Run a RAG query =======================================================
const userQuery = "How do I roll back a Kubernetes deployment?";

const retrieved = retrieve(userQuery, DOCS);

if (!retrieved) {
  console.log("No relevant document found for query:", userQuery);
} else {
  const prompt =
    "Answer the question below using ONLY the information in <context>. " +
    "If the answer is not there, say so.\n\n" +
    "<context>\n" + retrieved.text + "\n</context>\n\n" +
    "Question: " + userQuery;

  const answer = await getCompletion(prompt);
  console.log("Retrieved doc:", retrieved.id);
  console.log("Answer:", answer);
}

## Going Further: Web Search, Fetch, and the Files API

The in cell docs array above is a teaching aid. For real retrieval needs, Anthropic provides three complementary capabilities:

* **`web_search` tool**: a built in server side tool that lets Claude search the web during a message turn. Declare it in the `tools` array as `{ type: "web_search_20260209", name: "web_search" }` for current models like Sonnet 4.6 (older models use the basic `web_search_20250305`). Claude issues queries and Anthropic returns results with citations. Docs: https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search
* **`web_fetch` tool**: a separate built in server side tool that retrieves and reads the content of a specific URL already present in the conversation. Declare it as `{ type: "web_fetch_20260209", name: "web_fetch" }` (older models use `web_fetch_20250910`). Use it when you already know the page you want. Docs: https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-fetch
* **Files API**: upload large documents (PDFs, long text files) once and reference them by `file_id` across many requests, avoiding resending the same content each call. Upload via `POST /v1/files` (beta header `files-api-2025-04-14`) and reference with `{ type: "document", source: { type: "file", file_id: "..." } }`. Docs: https://platform.claude.com/docs/en/build-with-claude/files

No runnable cell is provided here, since these tools require an active API key and network access, but the pattern you'd use is the same: retrieve or load the content, inject it into `<context>` tags, and instruct Claude to answer from it.

## Reflect and Congratulations {#reflect}

### Reflect: keyword retrieval vs embeddings

The `retrieve()` above fails in predictable ways:

* **Synonyms**: a query about "reverting a deployment" scores zero against docs that say "rollback", because the tokens don't overlap.
* **Word order / phrasing**: keyword scoring ignores whether the terms appear together in a meaningful way.
* **Long queries**: more terms mean more noise; the score is dominated by common words unless you filter stop words.

Embedding based retrieval encodes meaning rather than tokens, so "rollback" and "revert" land near each other in vector space. For production RAG, start with a hybrid approach: semantic search for recall, keyword reranking for precision.

Other failure modes to keep in mind:
* **Chunk size matters**: too small and you lose context; too large and you dilute the relevant signal.
* **Top K vs threshold**: returning a fixed number of docs can include irrelevant ones; a similarity threshold is more precise but needs tuning.
* **Context stuffing**: pasting many large chunks can push the key information far from the question. Use Chapter 8's placement advice and consider reranking to put the best chunk closest to the query.

***

### Congratulations, you've completed the tutorial!

You've worked through every chapter: from basic prompting and formatting all the way through role play, long context, evals, tool use, chaining, and now search and retrieval. These techniques compose, so combine RAG with chaining (Chapter 10.1), tool use (Chapter 10.2), and the grounding strategies from Chapter 8 to build production grade LLM features.

Head back to the [README](../README.md) for a summary of all chapters and links to further reading.